# 1. CONFIGURACIÓN Y EXPLORACIÓN DE DATOS

In [0]:
# -------------------- PASO 1: VERIFICACIÓN INICIAL --------------------

print("Notebook listo")

In [0]:
# -------------------- PASO 2: INSPECCIÓN DEL DIRECTORIO --------------------

import os

print("Directorio actual:")
print(os.getcwd())

print("\nContenido del directorio:")
print(os.listdir("."))

In [0]:
# -------------------- PASO 2: INSPECCIÓN DEL DIRECTORIO --------------------

import os

print("Directorio actual:")
print(os.getcwd())

print("\nContenido del directorio:")
print(os.listdir("."))

In [0]:
# -------------------- PASO 3: VERIFICACIÓN DE ARCHIVOS DE ENTRADA --------------------

import os

ruta_proyecto = "/Workspace/Users/oliveraluis2112@gmail.com/proyecto-etl-delivery"

for carpeta in ["restaurants.csv", "restaurant-menus.csv"]:
    ruta = os.path.join(ruta_proyecto, carpeta)

    print(f"\nCarpeta: {carpeta}")
    print("Es carpeta:", os.path.isdir(ruta))

    if os.path.isdir(ruta):
        archivos = os.listdir(ruta)
        print("Cantidad de elementos:", len(archivos))
        print("Primeros 10 elementos:", archivos[:10])
    else:
        print("Es archivo:", os.path.isfile(ruta))

In [0]:
# -------------------- PASO 4: EXPLORACIÓN INICIAL DE LOS DATOS --------------------

import os
import pandas as pd

ruta_proyecto = "/Workspace/Users/oliveraluis2112@gmail.com/proyecto-etl-delivery"

# Ruta del archivo de restaurantes
ruta_restaurantes = os.path.join(
    ruta_proyecto, "restaurants.csv", "restaurants.csv"
)

# Mostrar estructura y muestra de restaurantes
df_restaurantes_muestra = pd.read_csv(ruta_restaurantes, nrows=5)

print("RESTAURANTES")
print("Columnas:", df_restaurantes_muestra.columns.tolist())
display(df_restaurantes_muestra)

# Inspeccionar una parte del primer archivo de menús
ruta_menus = os.path.join(
    ruta_proyecto, "restaurant-menus.csv", "restaurant_menus_parte_1.csv"
)

df_menus_muestra = pd.read_csv(ruta_menus, nrows=5)

print("\nMENÚS")
print("Columnas:", df_menus_muestra.columns.tolist())
display(df_menus_muestra)

In [0]:
# -------------------- PASO 5: EXTRACCIÓN E IDENTIFICACIÓN DE FUENTES --------------------

import os
import glob
import pandas as pd

# Ruta principal del proyecto
ruta_proyecto = "/Workspace/Users/oliveraluis2112@gmail.com/proyecto-etl-delivery"

# 1. Cargar restaurantes
ruta_restaurantes = os.path.join(
    ruta_proyecto, "restaurants.csv", "restaurants.csv"
)

df_restaurantes = pd.read_csv(ruta_restaurantes)

print("RESTAURANTES")
print("Dimensiones:", df_restaurantes.shape)
print("Columnas:", df_restaurantes.columns.tolist())

# 2. Identificar los 10 archivos de menús
ruta_menus = os.path.join(
    ruta_proyecto, "restaurant-menus.csv", "restaurant_menus_parte_*.csv"
)

archivos_menus = sorted(
    glob.glob(ruta_menus),
    key=lambda ruta: int(
        os.path.basename(ruta).split("_parte_")[1].split(".")[0]
    )
)

print("\nArchivos de menús encontrados:", len(archivos_menus))
for archivo in archivos_menus:
    print(os.path.basename(archivo))

In [0]:
# -------------------- PASO 6: EVALUACIÓN DE ARCHIVOS DE MENÚS --------------------

import os
import pandas as pd

print("TAMAÑO DE LOS ARCHIVOS DE MENÚS")

for archivo in archivos_menus:
    tamaño_mb = os.path.getsize(archivo) / (1024 ** 2)
    print(f"{os.path.basename(archivo)}: {tamaño_mb:.2f} MB")

print("\nMUESTRA DEL PRIMER ARCHIVO")

df_menus_muestra = pd.read_csv(archivos_menus[0], nrows=5)

print("Dimensiones de la muestra:", df_menus_muestra.shape)
print("Tipos de datos:")
print(df_menus_muestra.dtypes)

display(df_menus_muestra)

# 2. TRANSFORMACIÓN Y LIMPIEZA DE DATOS

In [0]:
# -------------------- PASO 7: PROCESAMIENTO POR BLOQUES Y LIMPIEZA DE PRECIOS --------------------

import pandas as pd
import glob
import os

# Archivos ordenados de menús
archivos_menus = sorted(
    glob.glob(
        os.path.join(
            ruta_proyecto,
            "restaurant-menus.csv",
            "restaurant_menus_parte_*.csv"
        )
    ),
    key=lambda x: int(
        os.path.basename(x).split("_parte_")[1].split(".")[0]
    )
)

# Procesamiento por bloques
bloques_validos = []
filas_leidas = 0

for archivo in archivos_menus:
    print(f"Procesando: {os.path.basename(archivo)}")

    for bloque in pd.read_csv(
        archivo,
        chunksize=100_000,
        dtype={"price": "string"}
    ):
        filas_leidas += len(bloque)

        # Limpiar y convertir el precio
        precio = (
            bloque["price"]
            .str.replace(" USD", "", regex=False)
            .str.replace(",", ".", regex=False)
        )

        bloque["price"] = pd.to_numeric(precio, errors="coerce")

        # Conservar precios válidos y mayores que cero
        bloque = bloque[
            bloque["price"].notna() &
            (bloque["price"] > 0)
        ]

        if not bloque.empty:
            bloques_validos.append(bloque)

print("\nTotal de filas leídas:", filas_leidas)
print("Bloques válidos almacenados:", len(bloques_validos))

In [0]:
# -------------------- PASO 8: CONSOLIDACIÓN Y VALIDACIÓN DE MENÚS --------------------

# Unir los bloques válidos en un solo DataFrame
df_menus = pd.concat(bloques_validos, ignore_index=True)

print("Dimensiones de df_menus:", df_menus.shape)

print("\nTipos de datos:")
print(df_menus.dtypes)

print("\nPrimeras 5 filas:")
display(df_menus.head())

print("\nPrecios:")
print(df_menus["price"].describe())

In [0]:
# -------------------- PASO 9: LIMPIEZA Y TRANSFORMACIÓN DE RESTAURANTES --------------------

# Crear una copia para conservar los datos originales
df_restaurantes = df_restaurantes.copy()

# 1. Transformar el rango de precios
mapa_precios = {
    "$": "Económico",
    "$$": "Moderadamente caro",
    "$$$": "Caro",
    "$$$$": "Muy caro"
}

df_restaurantes["price_range"] = (
    df_restaurantes["price_range"]
    .map(mapa_precios)
)

# 2. Limpiar y separar la dirección
direccion = (
    df_restaurantes["full_address"]
    .fillna("")
    .str.split(",", n=2, expand=True)
)

df_restaurantes["calle"] = direccion[0].str.strip()

df_restaurantes["ciudad"] = (
    direccion[1].str.strip()
    if 1 in direccion.columns
    else ""
)

df_restaurantes["estado_zip"] = (
    direccion[2].str.strip()
    if 2 in direccion.columns
    else ""
)

# 3. Eliminar restaurantes sin puntuación o con puntuación cero
df_restaurantes = df_restaurantes[
    df_restaurantes["score"].notna()
    & (df_restaurantes["score"] > 0)
].copy()

# 4. Convertir categorías a minúsculas
df_restaurantes["category"] = (
    df_restaurantes["category"].str.lower()
)

# 5. Revisar resultados
print("Dimensiones después de transformar:", df_restaurantes.shape)

display(
    df_restaurantes[
        [
            "name",
            "score",
            "category",
            "price_range",
            "calle",
            "ciudad",
            "estado_zip"
        ]
    ].head(10)
)

In [0]:
# -------------------- PASO 10: AUDITORÍA DE RANGOS DE PRECIOS ORIGINALES --------------------

# Leer únicamente la columna original de rangos de precios
ruta_restaurantes = os.path.join(
    ruta_proyecto,
    "restaurants.csv",
    "restaurants.csv"
)

precios_originales = pd.read_csv(
    ruta_restaurantes,
    usecols=["price_range"]
)

print("Valores originales de price_range:")
display(
    precios_originales["price_range"]
    .value_counts(dropna=False)
    .rename_axis("valor_original")
    .reset_index(name="cantidad")
)

In [0]:
# -------------------- PASO 11: NORMALIZACIÓN DE CATEGORÍAS DE PRECIOS --------------------

# Definir las categorías de precios
mapa_precios = {
    "$": "Económico",
    "$$": "Moderadamente caro",
    "$$$": "Caro",
    "$$$$": "Muy caro"
}

# Recuperar el valor original desde el CSV
precios_originales = pd.read_csv(
    ruta_restaurantes,
    usecols=["price_range"]
)

# Aplicar el mapeo y asignar los resultados a los restaurantes actuales
df_restaurantes["price_range"] = (
    precios_originales.loc[df_restaurantes.index, "price_range"]
    .map(mapa_precios)
    .fillna("Sin clasificar")
)

# Verificar los resultados
display(
    df_restaurantes["price_range"]
    .value_counts(dropna=False)
    .rename_axis("categoría_precio")
    .reset_index(name="cantidad")
)

In [0]:
# -------------------- PASO 12: FILTRADO POR CANTIDAD DE VALORACIONES --------------------

# Filtrar restaurantes con más de 100 valoraciones
df_restaurantes_filtrados = df_restaurantes[
    df_restaurantes["ratings"].notna() &
    (df_restaurantes["ratings"] > 100)
].copy()

print("Restaurantes antes del filtro:", len(df_restaurantes))
print("Restaurantes con más de 100 valoraciones:", len(df_restaurantes_filtrados))

display(df_restaurantes_filtrados.head())

In [0]:
# -------------------- PASO 13: INTEGRACIÓN DE RESTAURANTES Y MENÚS --------------------

# Unir restaurantes filtrados con sus menús
df_master = pd.merge(
    df_restaurantes_filtrados,
    df_menus,
    left_on="id",
    right_on="restaurant_id",
    how="inner",
    suffixes=("_restaurante", "_menu")
)

print("Dimensiones de df_master:", df_master.shape)
display(df_master.head())

In [0]:
# -------------------- PASO 14: VALIDACIÓN ESTRUCTURAL DE DF_MASTER --------------------

print("Dimensiones de df_master:", df_master.shape)

print("\nColumnas de df_master:")
print(df_master.columns.tolist())

In [0]:
%pip install sqlalchemy

# 3. CARGA Y VALIDACIÓN DE DATOS

In [0]:
# -------------------- PASO 15: PREPARACIÓN DE LA CARGA EN SQLITE --------------------

# LOAD: importar librerías para la base de datos
import sqlite3
from sqlalchemy import create_engine

print("Librerías para SQLite y SQLAlchemy importadas correctamente.")

In [0]:
# -------------------- PASO 16: CONFIGURACIÓN DE LA BASE DE DATOS SQLITE --------------------

# LOAD: crear la base de datos SQLite
ruta_db = os.path.join(ruta_proyecto, "delivery_insights.db")

engine = create_engine(f"sqlite:///{ruta_db}")

print("Base de datos configurada en:", ruta_db)

In [0]:
# -------------------- PASO 17: CARGA DE DATOS EN SQLITE --------------------

# LOAD: guardar los datos en SQLite
df_master.to_sql(
    name="master_food_data",
    con=engine,
    if_exists="replace",
    index=False
)

print("Datos cargados correctamente en master_food_data.")

In [0]:
# -------------------- PASO 18: VALIDACIÓN DE REGISTROS CARGADOS EN SQLITE --------------------

# LOAD: verificar los registros guardados
with engine.connect() as conexion:
    resultado = conexion.execute(
        __import__("sqlalchemy").text(
            "SELECT COUNT(*) FROM master_food_data"
        )
    )
    total_registros = resultado.scalar()

print("Registros guardados en SQLite:", total_registros)
print("Registros en df_master:", len(df_master))

# 4. ANÁLISIS GEOGRÁFICO DE RESTAURANTES

In [0]:
# -------------------- PASO 19: TOP 5 CIUDADES POR CANTIDAD DE RESTAURANTES --------------------

# BI: Top 5 ciudades con más restaurantes
consulta_ciudades = """
SELECT
    ciudad,
    COUNT(DISTINCT id) AS cantidad_restaurantes
FROM master_food_data
WHERE ciudad IS NOT NULL
  AND TRIM(ciudad) <> ''
GROUP BY ciudad
ORDER BY cantidad_restaurantes DESC
LIMIT 5;
"""

df_top_ciudades = pd.read_sql_query(
    consulta_ciudades,
    engine
)

display(df_top_ciudades)

In [0]:
# -------------------- PASO 20: VISUALIZACIÓN DEL TOP 5 DE CIUDADES --------------------

# BI: Top 5 ciudades con mayor cantidad de restaurantes
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

datos = df_top_ciudades.sort_values(
    "cantidad_restaurantes",
    ascending=True
)

fig, ax = plt.subplots(figsize=(10, 5.8))

barras = ax.barh(
    datos["ciudad"],
    datos["cantidad_restaurantes"],
    height=0.62
)

# Etiquetas con los valores
ax.bar_label(
    barras,
    fmt="%.0f",
    padding=7,
    fontsize=10
)

# Título y ejes
ax.set_title(
    "Top 5 ciudades por cantidad de restaurantes",
    loc="left",
    fontsize=15,
    fontweight="bold",
    pad=18
)
ax.set_xlabel("Cantidad de restaurantes", fontsize=10, labelpad=10)
ax.set_ylabel("")

# Estilo limpio y profesional
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_xlim(0, datos["cantidad_restaurantes"].max() * 1.15)
ax.grid(axis="x", linestyle="--", alpha=0.25)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_alpha(0.3)

ax.tick_params(axis="y", labelsize=10, length=0)
ax.tick_params(axis="x", labelsize=9)

fig.tight_layout()

fig.savefig(
    "/Workspace/Users/oliveraluis2112@gmail.com/proyecto-etl-delivery/images/top_5_ciudades.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

# 5. ANÁLISIS DE PRECIOS

In [0]:
# -------------------- PASO 21: DISTRIBUCIÓN POR CATEGORÍA DE PRECIO --------------------

# BI: distribución por categoría de precio
consulta_precios = """
SELECT
    price_range AS categoria_precio,
    COUNT(DISTINCT id) AS cantidad_restaurantes
FROM master_food_data
GROUP BY price_range
ORDER BY cantidad_restaurantes DESC;
"""

df_precios = pd.read_sql_query(consulta_precios, engine)

display(df_precios)

In [0]:
# -------------------- PASO 23: PRECIO PROMEDIO DE PLATOS POR CIUDAD --------------------

# BI: precio promedio de platos en las 3 principales ciudades
consulta_precio_ciudades = """
SELECT
    ciudad,
    ROUND(AVG(price), 2) AS precio_promedio,
    COUNT(*) AS cantidad_platos
FROM master_food_data
WHERE ciudad IN (
    SELECT ciudad
    FROM master_food_data
    WHERE ciudad IS NOT NULL
      AND TRIM(ciudad) <> ''
    GROUP BY ciudad
    ORDER BY COUNT(DISTINCT id) DESC
    LIMIT 3
)
GROUP BY ciudad
ORDER BY COUNT(DISTINCT id) DESC;
"""

df_precio_ciudades = pd.read_sql_query(
    consulta_precio_ciudades,
    engine
)

display(df_precio_ciudades)

In [0]:
# -------------------- PASO 24: VISUALIZACIÓN DEL PRECIO PROMEDIO POR CIUDAD --------------------

# BI: comparación del precio promedio de platos por ciudad
import matplotlib.pyplot as plt

datos = df_precio_ciudades.sort_values(
    "precio_promedio",
    ascending=True
)

fig, ax = plt.subplots(figsize=(9, 4.8))

ax.scatter(
    datos["precio_promedio"],
    datos["ciudad"],
    s=110,
    zorder=3
)

# Etiquetas de precios
for _, fila in datos.iterrows():
    ax.annotate(
        f'USD {fila["precio_promedio"]:.2f}',
        (fila["precio_promedio"], fila["ciudad"]),
        xytext=(9, 0),
        textcoords="offset points",
        va="center",
        fontsize=10
    )

ax.set_title(
    "Precio promedio de platos por ciudad",
    loc="left",
    fontsize=14,
    fontweight="bold",
    pad=16
)

ax.set_xlabel("Precio promedio (USD)")
ax.set_ylabel("")
ax.set_xlim(8, 14)
ax.set_xticks([8, 9, 10, 11, 12, 13, 14])
ax.grid(axis="x", linestyle="--", alpha=0.25)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)

fig.savefig(
    "/Workspace/Users/oliveraluis2112@gmail.com/proyecto-etl-delivery/images/precio_promedio_ciudad.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

# 6. ANÁLISIS DE CALIDAD Y EXPERIENCIA DEL USUARIO

In [0]:
# -------------------- PASO 25: PUNTUACIÓN PROMEDIO POR CATEGORÍA DE PRECIO --------------------

# BI: puntuación promedio por categoría de precio
consulta_score_precios = """
SELECT
    price_range AS categoria_precio,
    ROUND(AVG(score), 2) AS puntuacion_promedio,
    COUNT(DISTINCT id) AS cantidad_restaurantes
FROM master_food_data
WHERE price_range IS NOT NULL
GROUP BY price_range
ORDER BY puntuacion_promedio DESC;
"""

df_score_precios = pd.read_sql_query(
    consulta_score_precios,
    engine
)

display(df_score_precios)

In [0]:

# -------------------- PASO 26: DISTRIBUCIÓN DE RESTAURANTES POR CATEGORÍA DE PRECIO --------------------

# BI: distribución de restaurantes por categoría de precio
import matplotlib.pyplot as plt

datos = df_precios.sort_values(
    "cantidad_restaurantes",
    ascending=False
)

fig, ax = plt.subplots(figsize=(8, 6))

ax.pie(
    datos["cantidad_restaurantes"],
    labels=datos["categoria_precio"],
    autopct="%1.1f%%",
    startangle=90,
    counterclock=False,
    pctdistance=0.7,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5}
)

ax.set_title(
    "Distribución de restaurantes por categoría de precio",
    loc="left",
    fontsize=14,
    fontweight="bold",
    pad=18
)

ax.axis("equal")
fig.tight_layout()

fig.savefig(
    "/Workspace/Users/oliveraluis2112@gmail.com/proyecto-etl-delivery/images/distribucion_restaurantes_categoria_precio.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [0]:

# -------------------- PASO 27: RELACIÓN ENTRE PRECIO Y CALIDAD --------------------

# BI: ¿un restaurante más caro garantiza una mejor experiencia?
import matplotlib.pyplot as plt

datos = df_score_precios.sort_values(
    "puntuacion_promedio",
    ascending=True
)

fig, ax = plt.subplots(figsize=(9, 5.5))

barras = ax.barh(
    datos["categoria_precio"],
    datos["puntuacion_promedio"],
    height=0.6
)

# Etiquetas con la puntuación promedio
ax.bar_label(
    barras,
    fmt="%.2f",
    padding=6,
    fontsize=10
)

ax.set_title(
    "¿Un precio más alto garantiza una mejor experiencia?",
    loc="left",
    fontsize=14,
    fontweight="bold",
    pad=18
)

ax.set_xlabel("Puntuación promedio (score)")
ax.set_ylabel("Categoría de precio")
ax.set_xlim(4.0, 5.0)
ax.grid(axis="x", linestyle="--", alpha=0.25)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)

fig.tight_layout()

fig.savefig(
    "/Workspace/Users/oliveraluis2112@gmail.com/proyecto-etl-delivery/images/puntuacion_promedio_categoria_precio.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()



## Conclusiones del análisis

Distribución geográfica: Milwaukee registra la mayor cantidad de restaurantes del conjunto analizado, con 122 establecimientos, seguida de Seattle con 119.

Categorías de precio: La categoría Económico concentra la mayor cantidad de restaurantes, con 450 establecimientos, seguida de Moderadamente caro, con 157.

Precio promedio de los platos: Seattle presenta el precio promedio más alto entre las tres ciudades principales, con USD 11.98, seguida de Lynnwood con USD 10.82 y Milwaukee con USD 9.53.

Puntuación por categoría de precio: Las categorías Moderadamente caro y Económico presentan puntuaciones promedio similares, de 4.62 y 4.61, respectivamente. Por tanto, en este conjunto de datos, una categoría de precio más alta no implica necesariamente una mejor puntuación.

Proceso ETL: Se cargaron, limpiaron y transformaron los datos de restaurantes y menús, se integraron mediante sus identificadores y se almacenaron 63,221 registros en la tabla master_food_data de SQLite para facilitar el análisis mediante consultas SQL.